# 飞机降落问题

**类别：** 调度

来源：[https://www.hexaly.com/templates/aircraft-landing-problem-2](https://www.hexaly.com/templates/aircraft-landing-problem-2)


## 问题描述

在**飞机降落问题**中,需要为一组飞机安排降落时间。每架飞机可以在目标时间附近的一个预定时间窗口内降落,且相邻两架飞机之间必须保持一定的间隔时间。每架飞机都需要支付提前或延迟降落的惩罚费用。目标是在满足所有约束条件的前提下,最小化总惩罚费用。

	

### 学习要点

- 使用 [list decision variable](https://www.hexaly.com/docs/last/mathematicaloperators/collectionvariables.html) 表示降落顺序
- 使用 [lambda 表达式](https://www.hexaly.com/docs/last/mathematicaloperators/delegates.html#) 递归地定义飞机的降落时间
- 使用非线性算子,如[三元条件表达式](https://www.hexaly.com/docs/last/modelerreference/expressions.html#conditional-ternary-expressions)


## 数据

我们提供来自 [OR Library](http://people.brunel.ac.uk/~mastjjb/jeb/orlib/airlandinfo.html) 的算例。数据文件的格式如下:

- 第一行:飞机数量与冻结时间
- 从第二行起,每架飞机 i 的信息:

- 出现时间
- 最早降落时间
- 目标降落时间
- 最晚降落时间
- 早于目标时间降落时,每单位时间的惩罚费用
- 晚于目标时间降落时,每单位时间的惩罚费用
- 对于每架飞机 j:在飞机 i 降落后,飞机 j 降落所需的间隔时间

我们仅考虑问题的静态版本,忽略出现时间与冻结时间。


## 建模方法

飞机降落问题的 Hexaly 模型使用 [list decision variable](https://www.hexaly.com/docs/last/mathematicaloperators/collectionvariables.html) 表示降落顺序。列表中的第 i 个元素对应第 i 个降落的飞机的索引。为了确保所有飞机均被调度,我们对列表变量施加排列约束。

我们还引入了另一个决策变量:每架飞机的偏好降落时间。当一架飞机在前一架飞机之后必须等待才能降落时,它只能在目标时间之后降落。因此,一架飞机的偏好降落时间在最早降落时间与目标时间之间。

[递归 lambda 函数](https://www.hexaly.com/docs/last/mathematicaloperators/delegates.html#special-case) 将每架飞机的降落时间定义为偏好降落时间与该飞机在不违反与前机间隔时间的前提下所能降落的最早时间之间的最大值。

目标是最小化由每架飞机提前或延迟降落所产生的惩罚费用。为了计算每架飞机的该费用,我们使用三元条件表达式,根据降落时间与目标时间的差值,选择提前费用或延迟费用。


## Python 实现


In [1]:
from optagent import ModelBuilder, solve


def read_elem(filename):
    with open(filename, encoding="utf-8") as f:
        return [str(elem) for elem in f.read().split()]


def read_instance(instance_file):
    file_it = iter(read_elem(instance_file))
    nb_planes = int(next(file_it))
    next(file_it)  # Skip freezeTime value
    earliest_time_data = []
    target_time_data = []
    latest_time_data = []
    earliness_cost_data = []
    tardiness_cost_data = []
    separation_time_data = []

    for p in range(nb_planes):
        next(file_it)  # Skip appearanceTime values
        earliest_time_data.append(int(next(file_it)))
        target_time_data.append(int(next(file_it)))
        latest_time_data.append(int(next(file_it)))
        earliness_cost_data.append(float(next(file_it)))
        tardiness_cost_data.append(float(next(file_it)))
        separation_time_data.append([0] * nb_planes)

        for pp in range(nb_planes):
            separation_time_data[p][pp] = int(next(file_it))

    return (
        nb_planes,
        earliest_time_data,
        target_time_data,
        latest_time_data,
        earliness_cost_data,
        tardiness_cost_data,
        separation_time_data,
    )


def solve_instance(
    nb_planes,
    earliest_time_data,
    target_time_data,
    latest_time_data,
    earliness_cost_data,
    tardiness_cost_data,
    separation_time_data,
    time_limit=20,
    output_file=None,
):
    model = ModelBuilder()

    # List variable: landing_order[p] is the index of the p-th plane to land
    landing_order = model.list(nb_planes, name="landing_order")

    # All planes must be scheduled
    model.constraint(model.count(landing_order) == nb_planes, name="all_planes_scheduled")

    # Create OptAgent arrays to be able to access them with an "at" operator
    target_time = model.array(target_time_data)
    latest_time = model.array(latest_time_data)
    earliness_cost = model.array(earliness_cost_data)
    tardiness_cost = model.array(tardiness_cost_data)
    separation_time = model.array(separation_time_data)

    # Int variable: preferred landing time for each plane
    preferred_time_vars = [
        model.int(default=earliest_time_data[p], lb=earliest_time_data[p], ub=target_time_data[p], name=f"preferred_time_{p}")
        for p in range(nb_planes)
    ]
    preferred_time = model.array(preferred_time_vars)

    # Landing time for each plane (recursive: respects separation with the previous plane)
    def landing_time_lambda(p, prev):
        return model.max(
            model.at(preferred_time, model.at(landing_order, p)),
            model.iif(
                p > 0,
                prev + model.at(
                    separation_time,
                    model.at(landing_order, p - 1),
                    model.at(landing_order, p),
                ),
                0,
            ),
        )

    landing_time = model.array(
        model.range(0, nb_planes),
        model.lambda_function(landing_time_lambda),
        initial=0,
    )

    total_cost_terms = []
    for p in range(nb_planes):
        plane_index = model.at(landing_order, p)

        # Constraint on latest landing time
        model.constraint(model.at(landing_time, p) <= model.at(latest_time, plane_index), name=f"latest_time_{p}")

        # Cost for each plane
        difference_to_target_time = model.abs(model.at(landing_time, p) - model.at(target_time, plane_index))
        unit_cost = model.iif(
            model.at(landing_time, p) < model.at(target_time, plane_index),
            model.at(earliness_cost, plane_index),
            model.at(tardiness_cost, plane_index),
        )
        total_cost_terms.append(unit_cost * difference_to_target_time)

    total_cost = model.sum(*total_cost_terms)

    # Minimize the total cost
    model.minimize(total_cost, name="total_cost")

    solution = solve(model, time_limit_s=float(time_limit))

    landing_sequence = [int(item) for item in solution.variable_values[landing_order.node_id]]
    print(f"Nb planes = {nb_planes}; Total cost = {int(solution.objective_value)}; Status = {solution.status.value}")
    print("Landing order:", " ".join(str(p) for p in landing_sequence))

    if output_file is not None:
        with open(output_file, "w", encoding="utf-8") as f:
            f.write(f"{int(solution.objective_value)}\n")
            f.write(" ".join(str(p) for p in landing_sequence) + "\n")
    return solution


def main(instance_file, output_file=None, time_limit=20):
    return solve_instance(*read_instance(instance_file), time_limit=time_limit, output_file=output_file)


# if __name__ == "__main__":
#     if len(sys.argv) < 2:
#         print("Usage: python aircraft_landing.py instance_file [output_file] [time_limit]")
#         sys.exit(1)
#     instance_file = sys.argv[1]
#     output_file = sys.argv[2] if len(sys.argv) >= 3 else None
#     time_limit = int(sys.argv[3]) if len(sys.argv) >= 4 else 20
#     main(instance_file, output_file, time_limit)


## 运行实例

Notebook 直接调用 `main` 并显式传入实例路径。以下三段代码相互独立,可以根据需要单独运行;调整 `time_limit` 可以控制每个实例的求解时间。

In [2]:
from pathlib import Path

INSTANCE_DIR = Path.cwd() / "instances"
print("Instances:", INSTANCE_DIR)

Instances: /Users/dongbox/work/opt-agent/examples/examples/hexaly/aircraft_landing_problem/instances


In [3]:
solution_airland1 = main(INSTANCE_DIR / "airland1.txt", time_limit=1)

Starting OptAgent PORTFOLIO
Parameters: time_limit=1s threads=auto seed=0
Solve summary:
  status: FEASIBLE
  objective: 2770
  improvements: initial=5 search=7
  evaluated: 240
  wall_time: 1.00279s
  termination: wall_time_exhausted


Nb planes = 10; Total cost = 2770; Status = feasible
Landing order: 2 3 4 6 8 5 0 9 7 1


In [ ]:
solution_airland5 = main(INSTANCE_DIR / "airland5.txt", time_limit=1)

Starting OptAgent PORTFOLIO
Parameters: time_limit=10s threads=auto seed=0
Solve summary:
  status: FEASIBLE
  objective: 11390
  improvements: initial=2 search=34
  evaluated: 1504
  wall_time: 10.0002s
  termination: wall_time_exhausted


Nb planes = 20; Total cost = 11390; Status = feasible
Landing order: 4 2 5 3 8 6 9 17 16 7 18 13 12 11 0 1 19 14 15 10


In [ ]:
from optagent import ModelBuilder, solvedef read_elem(filename):    with open(filename, encoding="utf-8") as f:        return [str(elem) for elem in f.read().split()]def read_instance(instance_file):    """Read an aircraft landing instance.    Returns per-plane data and the separation-time matrix (a list of lists).    """    file_it = iter(read_elem(instance_file))    nb_planes = int(next(file_it))    next(file_it)  # Skip freezeTime value    earliest_time_data = []    target_time_data = []    latest_time_data = []    earliness_cost_data = []    tardiness_cost_data = []    separation_time_data = []    for p in range(nb_planes):        next(file_it)  # Skip appearanceTime values        earliest_time_data.append(int(next(file_it)))        target_time_data.append(int(next(file_it)))        latest_time_data.append(int(next(file_it)))        earliness_cost_data.append(float(next(file_it)))        tardiness_cost_data.append(float(next(file_it)))        separation_time_data.append([0] * nb_planes)        for pp in range(nb_planes):            separation_time_data[p][pp] = int(next(file_it))    return (        nb_planes,        earliest_time_data,        target_time_data,        latest_time_data,        earliness_cost_data,        tardiness_cost_data,        separation_time_data,    )def solve_instance(    nb_planes,    earliest_time_data,    target_time_data,    latest_time_data,    earliness_cost_data,    tardiness_cost_data,    separation_time_data,    time_limit=20,    output_file=None,):    model = ModelBuilder()    # List variable: landing_order[p] is the index of the p-th plane to land.    # ``default`` covers all planes so this is a full permutation out of the box,    # and we add an explicit constraint that fixes its cardinality.    landing_order = model.list(        nb_planes, default=tuple(range(nb_planes)), name="landing_order"    )    model.constraint(model.count(landing_order) == nb_planes, name="all_planes_scheduled")    # OptAgent arrays for parameter lookups    target_time = model.array(target_time_data)    latest_time = model.array(latest_time_data)    earliness_cost = model.array(earliness_cost_data)    tardiness_cost = model.array(tardiness_cost_data)    # Per-plane preferred landing time (an integer in [earliest, target])    preferred_time = [        model.int(            default=earliest_time_data[p],            lb=earliest_time_data[p],            ub=target_time_data[p],            name=f"preferred_time_{p}",        )        for p in range(nb_planes)    ]    preferred_time_array = model.array(preferred_time)    # Landing time for each schedule position.    # We avoid the OptAgent 2D ``at(separation_time, at(landing_order, p-1), at(landing_order, p))``    # lookup, which is not supported by the current kernel, by encoding the separation    # constraint pair-by-pair in Python over the list variable.    landing_time = [        model.int(default=earliest_time_data[p], lb=0, ub=10**9, name=f"landing_time_{p}")        for p in range(nb_planes)    ]    total_cost_terms = []    for p in range(nb_planes):        plane_index = model.at(landing_order, p)        # Lower bound from preferred time        model.constraint(            landing_time[p] >= model.at(preferred_time_array, plane_index),            name=f"preferred_{p}",        )        # Latest landing time        model.constraint(            landing_time[p] <= model.at(latest_time, plane_index),            name=f"latest_{p}",        )        # Cost: earliness if before target time, tardiness otherwise        difference_to_target_time = model.abs(landing_time[p] - model.at(target_time, plane_index))        unit_cost = model.iif(            landing_time[p] < model.at(target_time, plane_index),            model.at(earliness_cost, plane_index),            model.at(tardiness_cost, plane_index),        )        total_cost_terms.append(unit_cost * difference_to_target_time)    # Pairwise separation constraints between consecutive positions in the schedule.    # For every pair of consecutive schedule positions (p-1, p) and every pair of    # planes (a, b), enforce: if landing_order[p-1] == a and landing_order[p] == b    # then landing_time[p] >= landing_time[p-1] + separation[a][b].    # We materialise this by iterating over (p, a, b) and using the kernel's    # supported ``at(landing_order, p)`` lookups directly (1D, two dynamic indices,    # one for each schedule position).    for p in range(1, nb_planes):        for a in range(nb_planes):            for b in range(nb_planes):                prev_is_a = model.at(landing_order, p - 1) == a                curr_is_b = model.at(landing_order, p) == b                both = model.and_(prev_is_a, curr_is_b)                sep = separation_time_data[a][b]                # If both positions match (a, b), enforce the separation gap                needed_gap = landing_time[p - 1] + sep                constraint = model.or_(model.not_(both), landing_time[p] >= needed_gap)                model.constraint(constraint, name=f"sep_{p}_{a}_{b}")    total_cost = model.sum(*total_cost_terms)    model.minimize(total_cost, name="total_cost")    solution = solve(model, time_limit_s=float(time_limit))    landing_sequence = [int(item) for item in solution.variable_values[landing_order.node_id]]    print(        f"Nb planes = {nb_planes}; Total cost = {int(solution.objective_value)}; "        f"Status = {solution.status.value}"    )    print("Landing order:", " ".join(str(p) for p in landing_sequence))    if output_file is not None:        with open(output_file, "w", encoding="utf-8") as f:            f.write(f"{int(solution.objective_value)}\n")            f.write(" ".join(str(p) for p in landing_sequence) + "\n")    return solutiondef main(instance_file, output_file=None, time_limit=20):    return solve_instance(*read_instance(instance_file), time_limit=time_limit, output_file=output_file)# if __name__ == "__main__":#     if len(sys.argv) < 2:#         print("Usage: python aircraft_landing.py instance_file [output_file] [time_limit]")#         sys.exit(1)#     instance_file = sys.argv[1]#     output_file = sys.argv[2] if len(sys.argv) >= 3 else None#     time_limit = int(sys.argv[3]) if len(sys.argv) >= 4 else 20#     main(instance_file, output_file, time_limit)

## 运行实例Notebook 直接调用 `main` 并显式传入实例路径。以下三段代码相互独立,可以根据需要单独运行;调整 `time_limit` 可以控制每个实例的求解时间。

In [ ]:
from pathlib import PathINSTANCE_DIR = Path.cwd() / "instances"print("Instances:", INSTANCE_DIR)

In [ ]:
solution_airland1 = main(INSTANCE_DIR / "airland1.txt", time_limit=10)

In [ ]:
solution_airland5 = main(INSTANCE_DIR / "airland5.txt", time_limit=10)